# PanAf Ape Detection — Phase 1 "See"

**Pretrained MegaDetector V6 over PanAf500, on a Colab GPU.**

## How to run this

1. **Runtime → Change runtime type → GPU → Save.** (A100 if you have it.) Do this *first* —
   changing it later restarts the session.
2. **Runtime → Run all.** Approve the Google Drive popup when it appears.
3. **If it disconnects: Runtime → Run all again.** Nothing to edit, nothing to remember. Finished
   clips are skipped and the progress bar picks up where it stopped.

Repeat step 3 as many times as it takes. No files to upload, nothing to configure.

### Why a disconnect is survivable

Colab recycles the VM on a long run and wipes `/content` with it. So outputs go to **Drive**
(via `PANAF_ARTIFACTS_DIR`) before the run starts, and every clip is detected independently. A
disconnect therefore costs the video re-download — about 15 minutes — and **never the GPU work**,
which is the ~100-minute part.

You will see this, and the number only ever goes up:

```
[########................................] 100/500 clips detected  <- before this run
Resuming: 100 clip(s) are already done and will be skipped.
```

### What it does

- **Sections 1–9** (~15 min): 10 purposively chosen clips, MegaDetector every frame, ByteTrack,
  annotated video, and accuracy against the dataset's ground truth.
- **Section 11** (~2 h): all 500 clips, detector-only, then the tracking validation. **On by
  default** — it is the run this project currently needs.
- **Section 10** is off. That variant question was settled in commit `c81115f`; it is kept only so
  the decision stays reproducible.

### Three things to know

- **The device.** PyTorch-Wildlife accepts `device="cuda"`, stores it, and **never applies it** —
  the weights load on CPU and nothing raises. This notebook forces and *verifies* the placement, so
  watch for `weights forced onto cuda:0`.
- **Green vs amber.** In the annotated video, **green = MegaDetector prediction**,
  **amber = dataset ground truth** with its behaviour label. MegaDetector only outputs `animal` —
  not species, not behaviour.
- **Keep the tab open and click on it now and then.** Colab disconnects idle browsers, and that is
  the most common cause of a lost session.

### Licence

PanAf20K is under a **Non-Commercial Government Licence v2**. Clips downloaded here must not be
redistributed, and the annotated video is a derived work.

## 1. Check the GPU

In [ ]:
import subprocess

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode == 0:
    wanted = ("MiB", "Tesla", "NVIDIA")
    print([line for line in out.stdout.splitlines() if any(w in line for w in wanted)][:2])
    print("\nGPU OK.")
else:
    raise SystemExit("NO GPU. Runtime > Change runtime type > GPU > Save, then Runtime > Run all.")

## 2. Get the code

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/adikothuri3/PanAF-Ape-Detection.git"
REPO_DIR = "/content/PanAF-Ape-Detection"


def _git(*args: str) -> None:
    """Run git and raise on failure -- `!git` would not stop Run all."""
    result = subprocess.run(["git", *args], text=True)
    if result.returncode != 0:
        raise RuntimeError(f"`git {' '.join(args)}` failed. Read the output above.")


if not Path(REPO_DIR).exists():
    _git("clone", "--depth", "1", REPO_URL, REPO_DIR)
else:
    # A shallow clone cannot fast-forward a rewritten history; fetch and reset
    # rather than pull, so a re-run always lands on current main.
    _git("-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main")
    _git("-C", REPO_DIR, "reset", "--hard", "origin/main")

if not Path(REPO_DIR, "pyproject.toml").is_file():
    raise SystemExit(f"clone failed -- {REPO_DIR} has no pyproject.toml. Check the output above.")

os.chdir(REPO_DIR)
os.environ["PANAF_REPO_ROOT"] = REPO_DIR

# Section 11 redirects artifacts to Drive by setting this. It lives in the
# kernel, not the VM, so a "Run all" after a *browser* disconnect would still
# have it set -- and the 10-clip demo below would then write its tracked,
# confidence-0.20 detections into the 500-clip cache. Those clips would look
# "already done" to the full run and be skipped, silently poisoning it.
os.environ.pop("PANAF_ARTIFACTS_DIR", None)

print("working in", Path.cwd())

In [ ]:
# Colab's `!command` does NOT stop "Run all" when a command fails -- you get a
# confusing error several cells later instead of the real one. `run()` raises,
# so the notebook halts at the actual failure.
import os
import subprocess
import sys


def run(*args: str) -> None:
    """Run a command, streaming its output live, and raise if it fails.

    Streams line by line through a pipe rather than letting the child inherit
    this process's stdout. Two reasons, both learned the hard way on a long run:

    * Python block-buffers stdout when it is a pipe, so a subprocess printing
      steadily for an hour can show **nothing** until it exits -- which is
      indistinguishable from a hang. `PYTHONUNBUFFERED` plus an explicit read
      loop makes progress appear as it happens.
    * Merging stderr into stdout keeps log lines in order with printed output.
      They are separate streams with separate buffering, so leaving them apart
      shows a misleading sequence.
    """
    print("$", " ".join(args), flush=True)

    environment = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with subprocess.Popen(
        args,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=environment,
    ) as process:
        for line in process.stdout or ():
            print(line, end="", flush=True)

    if process.returncode != 0:
        raise RuntimeError(
            f"`{' '.join(args)}` failed with exit code {process.returncode}. "
            "Read the output above -- that is the real error. Do not continue past this cell."
        )

## 3. Install

**Deliberately does not install `requirements-colab.txt` here.** That file pins the full 170-package
locked environment including `torch`, and forcing it onto Colab replaces the CUDA-matched torch that
is already installed — a multi-gigabyte download that can leave the runtime without working CUDA.

Instead this installs only what Colab lacks and keeps Colab's torch. The locked file remains the
source of truth for reproducing the environment *outside* Colab.

Two or three minutes.

In [ ]:
# Check Python BEFORE pip runs. If Colab's interpreter is outside the range this
# project declares, `pip install -e .` refuses outright with "requires a
# different Python" -- and because the other three packages install fine, the
# only symptom is `panaf_ape_detection` missing several lines later. Catch it
# here, where the message can say what actually happened.
import sys
import tomllib
from pathlib import Path

required = tomllib.loads(Path("pyproject.toml").read_text())["project"]["requires-python"]
running = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
print(f"Python {running}   pyproject requires {required}")

try:
    from packaging.specifiers import SpecifierSet
    from packaging.version import Version

    supported = Version(f"{sys.version_info.major}.{sys.version_info.minor}") in SpecifierSet(
        required
    )
except ImportError:  # packaging absent -- skip rather than guess
    supported = True
    print("(packaging not installed; skipping the version check)")

if not supported:
    raise SystemExit(
        f"Colab is running Python {running}, which is outside this project's "
        f"declared range {required}. `pip install -e .` will refuse, and the only visible "
        "symptom would be a missing panaf_ape_detection.\n\n"
        "Options: pick a Colab runtime with a supported Python, or widen requires-python in "
        "pyproject.toml if the dependencies genuinely support this version -- do not widen it "
        "blind, the constraint is there because the lockfile was resolved against it."
    )

In [ ]:
# Keep Colab's CUDA-matched torch; add only what is missing.
# setuptools<81 first, because yolov5 (pulled in by PytorchWildlife) still
# imports pkg_resources, which setuptools 81 deprecated and 83 removed.
!pip install -q "setuptools<81"
!pip install -q pytorchwildlife soundfile librosa
!pip install -e . --no-deps   # not -q: if this fails, the reason must be visible

# `pip install -e` drops a .pth file into site-packages, but .pth files are only
# processed at interpreter *startup* -- so in a kernel that is already running,
# the package stays invisible to find_spec even though the install succeeded.
# Subprocesses start fresh and see it fine, which is why the pipeline itself
# works while the check below used to fail. Refresh the path instead of telling
# you to restart.
import importlib
import importlib.util  # `import importlib` alone does not bind .util
import site
import sys
from pathlib import Path

importlib.invalidate_caches()
site.main()  # re-process .pth files, including the one pip just wrote
src = str(Path(REPO_DIR) / "src")  # src layout: belt and braces
if src not in sys.path:
    sys.path.insert(0, src)

# Verify rather than assume -- a failed pip would otherwise surface as a
# confusing error several cells later.
REQUIRED = ("PytorchWildlife", "cv2", "supervision", "panaf_ape_detection")
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit(
        f"these did not install: {missing}. Read the pip output above -- that is the real "
        "error. If it mentions a version conflict, Runtime > Restart session, then "
        "Runtime > Run all (completed steps are skipped)."
    )
print("install OK:", ", ".join(REQUIRED))

In [ ]:
# Confirm torch still sees the GPU after the install.
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("\nCUDA is missing. Runtime > Restart session, then Runtime > Run all.")

## 4. Verify the stack

`smoke_inference.py` proves the heavy stack actually works — imports, ByteTrack, NumPy interop, a
video round-trip — **without downloading any model weights**. If something is wrong with the
environment, it fails here in seconds rather than twenty minutes into the run.

In [ ]:
run(sys.executable, "scripts/smoke_inference.py")

## 5. Download the clips

Straight from the Bristol deposit — no upload needed. ~23 MB.

Selection is **purposive, not random**: it profiles candidate annotations first (no video), then
greedily picks clips covering all nine behaviours, both species, crowded frames, small and large
subjects, and frames containing no ape. The reason for each pick goes into the manifest.

In [ ]:
run(sys.executable, "scripts/fetch_panaf500.py", "--count", "10", "--pool", "150")

In [ ]:
from pathlib import Path

import pandas as pd

manifest_path = Path("data/sample_manifest.csv")
if not manifest_path.is_file():
    raise SystemExit(
        "No manifest at data/sample_manifest.csv, which means the download cell above did not "
        "finish. Scroll up to it and read its output -- the real error is there. Re-running that "
        "cell is usually enough; network failures to data.bris.ac.uk are typically transient."
    )

manifest = pd.read_csv(manifest_path)
print(f"{len(manifest)} clips selected\n")
for _, row in manifest.iterrows():
    print(f"{row.clip_id}  [{row.split}]  {row.species}")
    print(f"    {row.selected_reason}\n")

## 6. Run detection and tracking on all 10 clips

Every frame of every clip: decode → MegaDetector → confidence filter → **ByteTrack** → compare
against ground truth → draw boxes → stitch to MP4 → write metrics and run metadata.

Tracking is CPU-side association over boxes the detector already produced, so it adds almost nothing
to the runtime. Two tables come out: detection accuracy, and track quality against the dataset's
`ape_id` — ID switches, fragmentation, and how much of each individual was followed.

**~10–15 minutes.** Watch for the device line early on:

```
WARNING ... PyTorch-Wildlife ignored device='cuda' (weights on 'cpu'); forcing it
INFO    ... weights forced onto cuda:0
```

That is the upstream bug being corrected. If it instead says the weights stayed on CPU, stop — the
run would be ~20x slower and the metadata would be wrong.

Clips already finished are skipped, so re-running after a dropped session resumes.

In [ ]:
run(sys.executable, "-m", "panaf_ape_detection.cli", "detect", "--config", "configs/colab.yaml")

## 7. Results

Real measurements at the stated confidence and IoU thresholds. A detection counts as correct when it
**localises** an annotated ape — MegaDetector cannot identify species, so no species claim is made.

In [ ]:
# Reading artifacts/ is done by panaf_ape_detection.reporting, not inline here.
# metrics/ holds detection metrics and metrics/tracking/ holds track metrics --
# two different schemas. A cell that globbed both and assumed one used to crash
# with KeyError: 'overall'. The loader knows the difference and is tested.
import pandas as pd

from panaf_ape_detection.reporting import (
    load_detection_metrics,
    pooled_counts,
    variants_in,
)

metrics = load_detection_metrics("artifacts")

# A run that stops part-way leaves some clips measured with the old model and
# some with the new. Every file is individually valid, so nothing else would
# show it -- and the pooled number would describe no model at all.
seen = variants_in(metrics)
if len(seen) > 1:
    raise SystemExit(
        f"artifacts/metrics/ mixes model variants: {sorted(seen)}. "
        "Re-run detection with --overwrite before reading these numbers."
    )
print(f"model: {sorted(seen)[0] or 'not recorded (pre-dates the field)'}\n")

display(
    pd.DataFrame(
        [
            {
                "clip": m["clip_id"],
                "frames": m["frames_evaluated"],
                "precision": round(m["overall"]["precision"], 3),
                "recall": round(m["overall"]["recall"], 3),
                "f1": round(m["overall"]["f1"], 3),
                "mean_iou": round(m["mean_iou"], 3),
                "empty_frames": m["empty_frames"],
                "FP_on_empty": m["false_positives_on_empty_frames"],
            }
            for m in metrics
        ]
    )
)

pooled = pooled_counts(metrics)
frames = sum(int(m["frames_evaluated"]) for m in metrics)
print(f"\n{len(metrics)} clips, {frames} frames")
print(f"TP={pooled.true_positives}  FP={pooled.false_positives}  FN={pooled.false_negatives}")
print(f"precision={pooled.precision:.4f}  recall={pooled.recall:.4f}  F1={pooled.f1:.4f}")
print(f"mean IoU={pooled.mean_iou:.4f}  (weighted by matched pairs, not averaged over clips)")

### Where it fails

This is the table that should drive any fine-tuning decision. A detector that misses arboreal
postures and small subjects needs different work from one that misses everything equally.

In [ ]:
from panaf_ape_detection.reporting import (
    format_recall_table,
    recall_by_behaviour,
    recall_by_size,
)

print("Recall by behaviour (worst first)")
print(format_recall_table(recall_by_behaviour(metrics)))

print("\nRecall by subject size (fraction of frame area)")
print(format_recall_table(recall_by_size(metrics), order=("small", "medium", "large")))
print("\nRead the size bands against the per-clip table above: if the bands are unevenly")
print("spread across clips of very different difficulty, this is a composition effect.")

### Track quality

Section 6 ran ByteTrack, so here is what it did — measured against the dataset's `ape_id`, not
against the tracker's own opinion of itself.

**Fragmentation** is predicted tracks per annotated ape: 1.00 is ideal, 0 means never tracked.
**Coverage** is the fraction of an individual's annotated frames that any track covered, and it is
capped by detection recall — a tracker cannot associate a box that was never produced.

**Identity coverage** is the number to watch. It is the fraction held by the ape's *single best*
track, so unlike coverage it falls when one animal is chopped into several tracks. The gap between
the two is the fragmentation tax.

**Purity** and **merges** exist to catch the opposite failure. Joining two apes into one track
removes the ID switches, drives fragmentation to the ideal 1.00, and leaves coverage untouched —
every other column here rewards it. Purity is the share of a track's frames belonging to one animal,
and it is the only column that notices.

**Jitter** is normalised frame-to-frame box shake, needing no ground truth: steady motion scores 0
however fast it is, so only genuine wobble counts. Lower is smoother.

In [ ]:
from panaf_ape_detection.reporting import load_track_metrics, pooled_track_metrics

tracks = load_track_metrics("artifacts")
if not tracks:
    print("Tracking was disabled for this run (tracking.enabled: false).")
else:
    # .get() on the identity columns: metrics written before those fields
    # existed still load, and must read as "not measured" rather than as zero.
    display(
        pd.DataFrame(
            [
                {
                    "clip": t["clip_id"],
                    "apes": t["annotated_individuals"],
                    "tracks": t["predicted_tracks"],
                    "ID switches": t["total_id_switches"],
                    "fragmentation": t["mean_fragmentation"],
                    "coverage": t["mean_coverage"],
                    "identity coverage": t.get("mean_identity_coverage"),
                    "purity": t.get("mean_track_purity"),
                    "merges": t.get("id_merges"),
                    "jitter": t.get("mean_jitter"),
                    "mostly tracked": t["mostly_tracked"],
                    "mostly lost": t["mostly_lost"],
                }
                for t in tracks
            ]
        )
    )

    # Not `pooled` -- that name holds the detection counts from section 7, and
    # rebinding it here would quietly change what a later cell reports.
    pooled_tracks = pooled_track_metrics(tracks)
    print(f"\n{pooled_tracks.individuals} individuals across {pooled_tracks.clips} clips")
    print(f"predicted tracks:  {pooled_tracks.predicted_tracks}")
    print(f"fragmentation:     {pooled_tracks.fragmentation:.2f}   per ape, 1.00 is ideal")
    print(f"ID switches:       {pooled_tracks.id_switches}")
    print(f"coverage:          {pooled_tracks.coverage:.4f}   any track")
    print(f"identity coverage: {pooled_tracks.identity_coverage:.4f}   single best track")
    print(f"track purity:      {pooled_tracks.track_purity:.4f}")
    print(f"tracks holding 2+ apes: {pooled_tracks.id_merges}")
    print(f"jitter:            {pooled_tracks.jitter:.4f}   box shake, lower is smoother")
    print(
        "\nCoverage minus identity coverage is the fragmentation tax: frames that were\n"
        "followed, but by a track that is not that ape's main one."
    )

## 8. Watch an annotated clip

**Green = MegaDetector prediction** (`#id` from the tracker, then confidence). **Amber = dataset ground truth** (with the
behaviour label and the individual's id). The legend is drawn on every frame, so a still pulled out
of the video is still unambiguous about which box came from where.

In [ ]:
import base64
from pathlib import Path

from IPython.display import HTML, display

videos = sorted(Path("artifacts/videos").glob("*_annotated.mp4"))
print(f"{len(videos)} annotated clips in artifacts/videos/\n")

# Colab's player cannot decode mp4v, so re-encode to H.264 just for display.
# Via run(), not `!ffmpeg`: a failed magic would not stop Run all, and the next
# line would then fail on a file ffmpeg never wrote -- a confusing error a long
# way from its cause.
for source in videos[:2]:
    playable = source.with_name(source.stem + "_h264.mp4")
    run(
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-i",
        str(source),
        "-vcodec",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        str(playable),
    )
    encoded = base64.b64encode(playable.read_bytes()).decode()
    print(source.name)
    display(
        HTML(
            f'<video width=720 controls><source src="data:video/mp4;base64,{encoded}" '
            f'type="video/mp4"></video>'
        )
    )

## 9. Keep the outputs

**Colab sessions are ephemeral — anything not copied out is lost.**

Set `USE_DRIVE = True` and re-run this cell to copy `artifacts/` to your Drive. You will be asked to
authorise access.

Do not commit the clips or the annotated video: they are derived works of a non-commercially
licensed dataset, and `artifacts/` and `data/` are git-ignored for that reason.

In [ ]:
USE_DRIVE = False
DESTINATION = "/content/drive/MyDrive/panaf-ape-detection/artifacts"

if USE_DRIVE:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    shutil.copytree("artifacts", DESTINATION, dirs_exist_ok=True)
    print("copied to", DESTINATION)
else:
    print("Drive copy off. Set USE_DRIVE = True and re-run this cell to keep the outputs.")

In [ ]:
# The run-metadata record: commit, config, verified device, variant, threshold,
# seed, input checksums, elapsed time. This is what makes the run reproducible.
# Selected by modification time -- once several experiments share the directory,
# the lexically last filename belongs to whichever experiment name sorts highest,
# not to the run that just finished.
from panaf_ape_detection.reporting import latest_run_metadata

meta = latest_run_metadata("artifacts")
if meta is None:
    print("No run metadata yet.")
else:
    for key in (
        "experiment_name",
        "git_commit",
        "git_dirty",
        "device",
        "model_variant",
        "confidence_threshold",
        "seed",
        "elapsed_seconds",
    ):
        print(f"{key:22} {meta.get(key)}")
    inputs = meta.get("inputs", [])
    print(f"{'inputs':22} {len(inputs) if isinstance(inputs, list) else 0} files, checksummed")

## 10. Variant comparison — `MDV6-yolov10-e` vs `MDV6-yolov9-c` (settled; off by default)

**This question is closed and this section does not run.**

`MDV6-yolov10-e` won and was adopted as the project default in commit `c81115f`. It is the variant
in `base.yaml`, `colab.yaml` and every config in this notebook — including the one section 6 just
used. Recall went **0.386 → 0.745 at unchanged precision**, for one line of config.

The section is kept so the decision stays *reproducible*, not because it is open. Re-running it
costs about 30 minutes of A100 time to re-derive a conclusion already recorded in
`reports/variant_comparison_2026-07-27.md`, so `RUN_VARIANT_COMPARISON` below defaults to `False`.

Set it to `True` only if you specifically want to reproduce that comparison from scratch.

**If you are here for the tracking work, skip to section 11.** Nothing there depends on this.

In [ ]:
RUN_VARIANT_COMPARISON = False  # settled in c81115f; ~30 min of A100 to reproduce

if RUN_VARIANT_COMPARISON:
    # BOTH arms, in one session. A fresh Colab has no artifacts/, so the losing
    # arm must be produced here too -- the comparison cells below read both.
    run(
        sys.executable,
        "-m",
        "panaf_ape_detection.cli",
        "detect",  # arm A: yolov9-c, the superseded variant
        "--config",
        "configs/colab-sweep-conf005.yaml",
        "--overwrite",
    )

    run(
        sys.executable,
        "-m",
        "panaf_ape_detection.cli",
        "detect",  # arm B: yolov10-e, the adopted one
        "--config",
        "configs/colab-variant-yolov10e.yaml",
        "--overwrite",
    )
else:
    print("Skipped: yolov10-e is already the project default (commit c81115f).")
    print("Result: recall 0.386 -> 0.745 at unchanged precision.")
    print("See reports/variant_comparison_2026-07-27.md, or set RUN_VARIANT_COMPARISON = True.")

### Compare the two variants

Both runs saved every detection down to 0.05, so this compares them at any threshold without
re-running inference. The last table is the one that decides it: **what fraction of each variant's
detections clear ByteTrack's 0.1 floor.**

In [ ]:
if RUN_VARIANT_COMPARISON:
    # Both arms saved every detection down to 0.05, so `evaluate --confidence`
    # re-scores them at any threshold without touching the GPU again.
    ARMS = {
        "yolov9-c  (superseded)": "configs/colab-sweep-conf005.yaml",
        "yolov10-e (adopted)": "configs/colab-variant-yolov10e.yaml",
    }

    for label, config in ARMS.items():
        print(f"\n{'=' * 60}\n{label}\n{'=' * 60}")
        for threshold in (0.05, 0.10, 0.20, 0.30):
            run(
                sys.executable,
                "-m",
                "panaf_ape_detection.cli",
                "evaluate",
                "--config",
                config,
                "--confidence",
                str(threshold),
            )
else:
    print("Skipped with the cell above.")

In [ ]:
# The question that decides adoption: do the new detections clear ByteTrack's
# floor? Detections at or below 0.1 are discarded by the tracker outright, and a
# new track needs activation + 0.1 -- so recall recovered below ~0.15 never
# reaches it. A variant that only finds fainter boxes changes nothing downstream.
from pathlib import Path

from panaf_ape_detection.reporting import score_bands

ROOTS = {
    "yolov9-c  (baseline)": "artifacts/sweep-conf005",
    "yolov10-e (new)": "artifacts/variant-yolov10e",
}

print(f"{'variant':22} {'total':>7} {'<=0.10':>8} {'0.10-0.15':>10} {'>0.15 usable':>14}")
for label, root in ROOTS.items():
    if not Path(root, "detections").is_dir():
        print(f"{label:22} not run yet")
        continue
    bands = score_bands(root)
    print(
        f"{label:22} {bands.total:7d} {bands.discarded:8d} {bands.cannot_start_track:10d} "
        f"{bands.usable:8d} ({bands.usable_fraction:5.0%})"
    )

### Keep the comparison outputs

**Section 9 ran before this comparison existed**, so its Drive copy did not include these results.
This cell saves everything again, after both arms. Colab sessions are ephemeral: the metrics JSONs
are what the write-up has to cite, so losing them means re-running 30 minutes of inference.

The small files are the ones that matter — `metrics/` and `metadata/`. `detections/` is worth
keeping too: it makes every future threshold sweep free.

In [ ]:
# Defaults to False, like USE_DRIVE above. It was True, and mounting Drive
# fails routinely -- the auth popup gets blocked, or times out. An exception
# here halts "Run all", so a failed *backup* silently prevented every later
# section from running at all. A backup step must never be able to do that.
SAVE_COMPARISON = False

if SAVE_COMPARISON:
    import shutil
    from pathlib import Path

    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print(f"Drive not mounted ({exc}). Nothing copied; the run itself is unaffected.")
    else:
        for arm in ("artifacts/sweep-conf005", "artifacts/variant-yolov10e"):
            # An arm that did not run leaves no directory; copytree would raise
            # FileNotFoundError and lose the arm that *did* run.
            if not Path(arm).is_dir():
                print(f"skipped {arm} -- not produced in this session")
                continue
            target = f"/content/drive/MyDrive/panaf-ape-detection/{arm}"
            shutil.copytree(arm, target, dirs_exist_ok=True)
            print("copied", arm, "->", target)
        print("\nBring metrics/ and metadata/ back into the repo to write the result up.")
else:
    print("Not saved. Set SAVE_COMPARISON = True and re-run this cell to keep the outputs.")

## Next

- Record what you saw in `experiments/experiment_log.md`, **including anything that failed**.
- Bring `metrics/` and `metadata/` from both arms back into the repo, so every number in the
  write-up traces to a file rather than to a screenshot of this notebook.
- **Read the comparison by the score distribution, not the recall row.** A variant that finds more
  faint boxes below 0.1 changes nothing downstream, because `sv.ByteTrack` discards them. A variant
  that lifts the same subjects above 0.15 raises track coverage, which is what caps Phase 2.
- If yolov10-e is no better, that is the result: capacity is not the gap, and the case for
  fine-tuning — or for different footage conditioning — is now evidence-backed rather than assumed.


## 11. The whole dataset — the run that makes tracking results mean something

Everything above uses 10 clips. That was the right size for building the pipeline and it is the
wrong size for **tuning** anything: settings chosen on 10 clips and then reported on the same 10
clips are not evidence, and these 10 were purposively chosen to be hard, so they are not a random
sample either.

PanAf500 is 500 clips, ~180,000 frames, and it ships its own `train` / `validation` / `test`
split, which the manifest records per row. That is what a tuning experiment should be divided by.

**This is also the ceiling.** PanAf20K is ~20,000 videos, but only this 500-clip subset carries
per-frame boxes and `ape_id` identities. Without identity ground truth there is nothing to score a
tracker against — ID switches and fragmentation are not merely expensive to compute there, they
are undefined. So 500 clips is not a compromise; it is the entire annotated universe for this task.

**Cost, from the measurement in the experiment log:** an A100 does 10 clips in 121 s with this
variant, so ~180,000 frames is **roughly 100 minutes**, plus ~1.1 GB of video to download. One
session. Annotated video is turned off for this run — 500 MP4s is gigabytes nobody watches and
costs a second decode of every clip.

Detection runs at **confidence 0.05**, which is not a claim that 0.05 is a good threshold (raw
precision there is 0.64). It is the right threshold to *cache* at: boxes below the threshold are
never written, so anything above 0.05 can be recovered later for free and nothing below it ever
can. The tracking work needs that low tail specifically.

Clips already finished are skipped, so a dropped session resumes where it stopped.

### First: send the outputs to Drive, or a disconnect costs you the whole run

A two-hour Colab run **will** be interrupted sometimes — idle timeout, a dropped browser, the VM
being recycled. When that happens `/content` is wiped, and everything the GPU produced goes with it.

The pipeline resumes cleanly per clip, but only if the outputs still exist. So point them at Drive
*before* starting. Then a disconnect costs at most the video downloads (~15 minutes to re-fetch),
never the detection work, which is the expensive part.

The next two cells do this: the first names the output directory, the second mounts Drive and
redirects it there via `PANAF_ARTIFACTS_DIR`.

The ~1.1 GB of downloaded video deliberately stays on the ephemeral disk — it is cheap to fetch
again and would otherwise fill your Drive. GPU time is the thing worth protecting.

In [ ]:
# Where the 500-clip run writes. The next cell redirects this to Drive; kept in
# its own cell so every later cell has the name even when Drive is unavailable.
if "run" not in globals():
    raise SystemExit(
        "The setup cells have not run in this session -- the runtime restarted, which "
        "also removed the installed package.\n\n"
        "Do Runtime > Run all. It redoes setup in about 4 minutes, and any clip already "
        "detected is skipped, so nothing is repeated."
    )

FULL500_ARTIFACTS = "artifacts/full500"

print(f"artifacts -> {FULL500_ARTIFACTS}")
print("Ephemeral. Run the next cell to put them on Drive instead.")

In [ ]:
SAVE_FULL_RUN_TO_DRIVE = True

if "FULL500_ARTIFACTS" not in globals():
    raise SystemExit("Run the cell above first (or Runtime > Run all).")

# PANAF_ARTIFACTS_DIR is an env override the config layer already honours, and
# subprocesses inherit it -- so setting it here redirects `detect`, `track` and
# `track-sweep` alike, with no config edit and no flags to remember.
DRIVE_ARTIFACTS = "/content/drive/MyDrive/panaf-ape-detection/artifacts/full500"

if SAVE_FULL_RUN_TO_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        # Never fatal: a failed *backup* must not stop the experiment. It only
        # means a disconnect would be expensive again.
        print(f"Drive not mounted ({exc}).")
        print(f"Continuing with ephemeral {FULL500_ARTIFACTS} -- a disconnect would lose the run.")
    else:
        FULL500_ARTIFACTS = DRIVE_ARTIFACTS
        os.environ["PANAF_ARTIFACTS_DIR"] = FULL500_ARTIFACTS
        Path(FULL500_ARTIFACTS).mkdir(parents=True, exist_ok=True)

        detections = Path(FULL500_ARTIFACTS, "detections")
        done = len(list(detections.glob("*.json"))) if detections.is_dir() else 0
        print(f"\nartifacts -> {FULL500_ARTIFACTS}")
        print(f"{done}/500 clips already detected. Anything done survives a disconnect.")
else:
    print(f"Drive off. Using {FULL500_ARTIFACTS}; a disconnect loses the run.")

In [ ]:
# On by default: this is the run the project needs, and leaving it off meant
# re-editing this line after every disconnect. Set False to skip section 11.
RUN_FULL_DATASET = True

if "FULL500_ARTIFACTS" not in globals():
    raise SystemExit("Run the two cells above first (or Runtime > Run all).")


def _progress(note: str) -> int:
    """Print how many clips are done, so a resumed run visibly picks up."""
    directory = Path(FULL500_ARTIFACTS, "detections")
    done = len(list(directory.glob("*.json"))) if directory.is_dir() else 0
    bar = "#" * (done * 40 // 500) + "." * (40 - done * 40 // 500)
    print(f"\n[{bar}] {done}/500 clips detected  <- {note}\n", flush=True)
    return done


if RUN_FULL_DATASET:
    before = _progress("before this run")
    if before:
        print(f"Resuming: {before} clip(s) are already done and will be skipped.\n")

    # --all takes every annotated clip in the deposit rather than a purposive
    # sample. Purposive selection exists to make a *small* sample representative;
    # taking everything makes it moot, and the dataset's own split is what a
    # tuning experiment should be divided by instead.
    #
    # The videos live on the ephemeral disk, so this re-downloads after a
    # restart -- about 15 minutes, against the ~100 minutes of GPU it protects.
    run(sys.executable, "scripts/fetch_panaf500.py", "--all")

    run(
        sys.executable,
        "-m",
        "panaf_ape_detection.cli",
        "detect",
        "--config",
        "configs/colab-full500.yaml",
    )

    after = _progress("after this run")
    if after < 500:
        print(f"{500 - after} clip(s) left. Re-run this cell -- it continues from here.")
    else:
        print("All 500 detected. Continue to the cells below.")
else:
    _progress("current state")
    print("Not running. Set RUN_FULL_DATASET = True above to start or continue.")

In [ ]:
# Integrity check on the cache, run after every attempt.
#
# A resumed run decides what to skip purely by whether a file exists, so a clip
# written by the *wrong* config is never corrected -- it is skipped, and the
# pooled numbers then quietly describe no single run. Every file is individually
# valid, so nothing else would reveal it.
from panaf_ape_detection.reporting import detection_cache_settings

settings = detection_cache_settings(FULL500_ARTIFACTS)

if not settings:
    print("Nothing detected yet.")
elif len(settings) == 1:
    only, count = next(iter(settings.items()))
    print(f"OK -- {count}/500 clips, all from the same settings:")
    print(f"    {only.describe()}")
    if count >= 500:
        print("\nComplete. Run the cells below for the tracking comparison.")
    else:
        print(f"\n{500 - count} to go. Runtime > Run all continues from here.")
else:
    print("STOP -- this cache is a mixture of runs and must not be used:\n")
    for found, count in sorted(settings.items(), key=lambda kv: -kv[1]):
        print(f"    {count:4d} clip(s)  {found.describe()}")
    print(
        "\nThe 500-clip run needs detector-only detections at confidence 0.05.\n"
        f"Delete the offending clips from {FULL500_ARTIFACTS}/detections/ and re-run,\n"
        "or delete that whole directory to start the cache cleanly."
    )

### Then sweep the tracker — no GPU needed

Once the cache above exists, tracking costs nothing to explore: detection is the expensive stage
and it is finished. `track-sweep` re-runs association over the saved boxes, so a hundred settings
take minutes on a laptop.

Arms are ranked by **identity coverage** — the share of each ape's annotated frames held by its
*single best* track. Plain coverage would reward chopping one animal into several tracks, and
counting ID switches alone would reward the opposite, merging two animals into one. Identity
coverage is pushed down by both, which is why it is the thing to maximise.

**`purity` and `merges` are printed beside it and are not decoration.** Every other tracking metric
improves when two apes are merged into one track: the switches vanish, fragmentation falls to the
ideal 1.00, coverage is unchanged. Purity is the only column that notices. Treat an arm that wins
while its merges climb as a failed arm, not a good one.

Measured throughput: **0.30 s per arm per 10 clips**, so a 108-arm grid over all 500 clips is
around 25 minutes with `-j 8`.

In [ ]:
if RUN_FULL_DATASET:
    # The validation itself: the same two settings, on clips they were never
    # tuned on. Both read the cache above, so neither costs GPU time.
    #
    # configs/base.yaml is the shipped tracker; configs/tracking-candidate.yaml
    # is what the 10-clip sweep produced. --metrics-dir keeps them apart, and
    # keeps both away from the 10-clip baseline.
    for label, cfg in (
        ("shipped", "configs/base.yaml"),
        ("candidate", "configs/tracking-candidate.yaml"),
    ):
        print(f"\n{'=' * 60}\n{label}\n{'=' * 60}")
        run(
            sys.executable,
            "-m",
            "panaf_ape_detection.cli",
            "track",
            "--config",
            cfg,
            "--detections-dir",
            f"{FULL500_ARTIFACTS}/detections",
            "--metrics-dir",
            f"{FULL500_ARTIFACTS}-{label}",
        )

    # Then check nothing beats it. If an arm does, the candidate was overfitted
    # to the 10 clips and this is where that shows up.
    run(
        sys.executable,
        "-m",
        "panaf_ape_detection.cli",
        "track-sweep",
        "--grid",
        "configs/sweeps/association.yaml",
        "--config",
        "configs/tracking-candidate.yaml",
        "--detections-dir",
        f"{FULL500_ARTIFACTS}/detections",
        "--jobs",
        "8",
    )
else:
    print("Needs the detection cache from the previous cell.")

In [ ]:
# The verdict. Reads whatever the cell above wrote; says so and stops if it has
# not been run, rather than failing.
from pathlib import Path

from panaf_ape_detection.reporting import load_track_metrics, pooled_track_metrics

ARMS = {name: f"{FULL500_ARTIFACTS}-{name}" for name in ("shipped", "candidate")}
present = {name: root for name, root in ARMS.items() if Path(root, "metrics").is_dir()}

if len(present) < 2:
    print(f"No results under {FULL500_ARTIFACTS}-*. Run the cells above first.")
else:
    rows = {name: pooled_track_metrics(load_track_metrics(root)) for name, root in present.items()}
    header = f"{'':24}" + "".join(f"{name:>14}" for name in rows)
    print(header)
    print("-" * len(header))
    for label, get in (
        ("identity coverage", lambda p: f"{p.identity_coverage:.4f}"),
        ("coverage", lambda p: f"{p.coverage:.4f}"),
        ("ID switches", lambda p: str(p.id_switches)),
        ("fragmentation", lambda p: f"{p.fragmentation:.2f}"),
        ("track purity", lambda p: f"{p.track_purity:.4f}"),
        ("tracks with 2+ apes", lambda p: str(p.id_merges)),
        ("jitter", lambda p: f"{p.jitter:.4f}"),
        ("mostly tracked", lambda p: str(p.mostly_tracked)),
        ("mostly lost", lambda p: str(p.mostly_lost)),
        ("individuals", lambda p: str(p.individuals)),
        ("clips", lambda p: str(p.clips)),
    ):
        print(f"{label:24}" + "".join(f"{get(p):>14}" for p in rows.values()))

    print(
        "\nThe candidate was tuned on 10 of these clips. If its margin here is much\n"
        "smaller than the 10-clip margin (identity coverage 0.6449 -> 0.7561), that\n"
        "gap is the overfitting, and it is the number worth reporting."
    )